In [67]:
# Detect if running in Google Colab
def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if in_colab():
    # Only runs in Colab, skipped on local
    # 1. Clone the repo and set paths
    REPO_URL = "https://github.com/MadKeyboardArtist/5703-Federated-Model.git"
    REPO_DIR = "/content/5703-Federated-Model"

    import os, sys
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    os.chdir(REPO_DIR)
    sys.path.append(REPO_DIR)
    print("CWD:", os.getcwd())

    # GPU check
    import torch, subprocess, textwrap
    print("CUDA available:", torch.cuda.is_available())
    !nvidia-smi

    # update the files
    %cd /content/5703-Federated-Model
    !git pull origin main

else:
    print("Running locally — skipping Colab setup.")


Running locally — skipping Colab setup.


In [68]:
# initialize model

# Per Round:
# server copies and sends the global model weights.
# client trains locally (on their local dataset).
# client returns its new weights and sample count.
# server aggregates the weights using FedAvg.
# global model is updated.
# (Optional) Evaluate global model on a held-out test set.


# heads record:
# 1. always save the newest
# 2. always save the best ever
# 3. always save curretn best

In [69]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [70]:
# external libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import importlib.util
import json
import shutil
import pandas as pd
import numpy as np

from collections import defaultdict

In [ ]:
# self-built files

# 1. model structure
from FederatedModel.federated_multihead_model import SharedEncoders, TabularClientModel, ImageClientModel, MultiClientModel
from FederatedModel.model_config import D_TABULAR, D_EMBEDDING, D_FUSION

# 2. site prediction functions
from FinalEvaluations.tabular_site_prediction import make_predictions as tabular_predictions
from FinalEvaluations.image_site_prediction   import make_predictions as image_predictions
# from multi_site_training   import training_loop as multi_predictions

# 3. evaluation metrics
from FinalEvaluations.evaluation_metrics import site_evaluation_cm    as site_cm_calculation
from FinalEvaluations.evaluation_metrics import site_evaluation_auroc as site_auroc_calculation
from FinalEvaluations.evaluation_metrics import site_evaluation_auprc as site_auprc_calculation
from FinalEvaluations.evaluation_metrics import site_evaluation_acc   as site_acc_calculation
from FinalEvaluations.evaluation_metrics import site_evaluation_f1    as site_f1_calculation

In [72]:
# reimport some self-built files if any changes
import importlib
import FederatedModel.federated_multihead_model
import FinalEvaluations.tabular_site_prediction
import FinalEvaluations.image_site_prediction

importlib.reload(FederatedModel.federated_multihead_model)
importlib.reload(FinalEvaluations.tabular_site_prediction)
importlib.reload(FinalEvaluations.image_site_prediction)

<module 'FinalEvaluations.image_site_prediction' from 'd:\\USYD\\2025 S2\\5703 Capstone\\model\\FinalEvaluations\\image_site_prediction.py'>

In [73]:
# basic confgs initialization
# outcomes saving
SAVED_MODELS_FOLDER = "SavedModels"
DATASETS_FOLDER = "Datasets"

buttom helpers

In [55]:
def load_transform_from_file(tsfm_file_path):
    module_name = os.path.splitext(os.path.basename(tsfm_file_path))[0]
    spec = importlib.util.spec_from_file_location(module_name, tsfm_file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.tsfm  # assumes each .py defines a variable `transform`

In [56]:
def extract_global_components(global_state, prefix):
    # Assume global_state is a full SharedEncoders.state_dict()
    # with keys like "tabular_enc.fc1.weight", "image_enc.conv1.weight", etc.

    enc_state = {
        k: v.clone()
        for k, v in global_state.items()
        if k.startswith(prefix) # keep only parameters start with prefix
    }

    return enc_state

In [57]:
def add_prefix_to_enc_dict (prefix, state_dict_list):
    result = []
    for state in state_dict_list:
        # Add the prefix to every key
        enc_with_prefix = {
            f"{prefix}{k}": v.clone() for k, v in state.items()
        }
        result.append(enc_with_prefix)
    return result

funtional helpers

In [ ]:
def assign_local_prediction_function (modality):
    if modality == "tabular":
        return tabular_predictions
    elif modality == "image":
        return image_predictions
        # return complete_image_training
    elif modality == "multi":
        pass
    else:
        # report ERROR
        return None

In [ ]:
'''
{'name': 'tabular_1',
  'modality': 'tabular',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'Datasets/tabular_1/diabetes_012_train.csv',
  'clean_dataset_val': 'Datasets/tabular_1/diabetes_012_val.csv',
  'clean_dataset_test': 'Datasets/tabular_1/diabetes_012_test.csv',
  'trained_local_head': 'SavedModelsoverall_best_local_headstabular_1.pth',
  'site_prediction': <function FinalEvaluations.tabular_site_evaluation.make_predictions(site_name, global_state, test_set_path, labelcol, n_classes, best_head_path)>
}
'''
def site_prediction (global_state, site):
    # tabular site prediction
    if site["modality"] == "tabular":
        site_name      = site["name"]
        test_set_path  = site["clean_dataset_test"]
        label_col_name = site["label_col"]
        num_of_classes = site["n_classes"]
        head_path      = site["trained_local_head"]

        return site["site_prediction"](
            site_name      = site_name,
            global_state   = global_state,           
            test_set_path  = test_set_path,
            labelcol       = label_col_name,
            n_classes      = num_of_classes,
            best_head_path = head_path
            )    
    
    # image site prediction
    elif site["modality"] == "image":
        site_name      = site["name"]
        test_set_path  = site["clean_dataset_test"]
        tsfm           = site["tsfm"]
        num_of_classes = site["n_classes"]
        head_path      = site["trained_local_head"]

        return site["site_prediction"](
            site_name      = site_name,
            global_state   = global_state,
            test_set_path  = test_set_path,
            tsfm           = tsfm,
            n_classes      = num_of_classes,
            best_head_path = head_path
            )    

    elif site["modality"] == "multi":
        # Not implemented yet
        return None

    else:
        print("site modality error")
        exit()

In [ ]:
# sites preparations:
def site_preparations (site):
    # 3.1 import all tsfm for image sites
    if site["modality"] == "image":
        tsfm_path = DATASETS_FOLDER + "/{:s}/{:s}_tsfm.py".format(site["name"], site["name"])
        full_path = tsfm_path
        site["tsfm"] = load_transform_from_file(full_path)

    # 3.2 get the saved local head
    model_name  = site["name"]
    modality    = site["modality"]
    n_classes   = site["n_classes"]

    # overall best site head 
    saved_head  = SAVED_MODELS_FOLDER + "overall_best_local_heads" + model_name + ".pth" 
    site["trained_local_head"] = saved_head

    # 3.3 assign the correct local training function
    site["site_prediction"] = assign_local_prediction_function(site["modality"])

    return site

evaluation start

In [62]:
# 0. (BEFORE training) find all recorded models
# overall best local heads
local_head_folder = SAVED_MODELS_FOLDER + "/overall_best_local_heads"
# best global encoders
global_enc_pth = SAVED_MODELS_FOLDER + "/best_global_encoders.pth"

In [ ]:
# 1. build global model
global_encoders = SharedEncoders(
    d_tabular   = D_TABULAR,
    d_embedding = D_EMBEDDING,
    d_fusion    = D_FUSION
    )
# dict to torch state
global_state = torch.load(global_enc_pth, map_location="cpu")

# torch to model (BUT do we need the global model here?)
global_encoders.load_state_dict(global_state, strict = False)
print("Global encoders loaded!")

Global encoders loaded!


In [64]:
# 2. import all sites (basic info)
# with open("sites_info_tabular.json", "r") as f:
with open("sites_info.json", "r") as f:
    sites_raw = json.load(f)
sites_raw

[{'name': 'tabular_1',
  'modality': 'tabular',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'Datasets/tabular_1/diabetes_012_train.csv',
  'clean_dataset_val': 'Datasets/tabular_1/diabetes_012_val.csv',
  'clean_dataset_test': 'Datasets/tabular_1/diabetes_012_test.csv'},
 {'name': 'image_1',
  'modality': 'image',
  'n_classes': 5,
  'clean_dataset_train': 'Datasets/image_1/train',
  'clean_dataset_val': 'Datasets/image_1/val',
  'clean_dataset_test': 'Datasets/image_1/test'}]

In [65]:
# 3. all sites preparations:
# FOR EACH SITE:
# 3.1 DATA: get testing dataset (DONE)
# 3.2 MODEL: find the head path (with site name)

sites = sites_raw.copy()
for site in sites_raw:
    site = site_preparations(site)

sites

[{'name': 'tabular_1',
  'modality': 'tabular',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'Datasets/tabular_1/diabetes_012_train.csv',
  'clean_dataset_val': 'Datasets/tabular_1/diabetes_012_val.csv',
  'clean_dataset_test': 'Datasets/tabular_1/diabetes_012_test.csv',
  'trained_local_head': 'SavedModelsoverall_best_local_headstabular_1.pth',
  'site_prediction': <function FinalEvaluations.tabular_site_evaluation.make_predictions(site_name, global_state, test_set_path, labelcol, n_classes, best_head_path)>},
 {'name': 'image_1',
  'modality': 'image',
  'n_classes': 5,
  'clean_dataset_train': 'Datasets/image_1/train',
  'clean_dataset_val': 'Datasets/image_1/val',
  'clean_dataset_test': 'Datasets/image_1/test',
  'tsfm': Compose(
      Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
      ColorJitter(

In [ ]:
# 4. operate federated training loop
# 4.0 configs
# (MANUALLY) set training rounds
pass

# (AOTU) decide components to update
for site in sites:
    if not TABULAR_UPDATE:
        if site["modality"] == "tabular":
            TABULAR_UPDATE = True
            continue
        else: pass

    if not IMAGE_UPDATE:
        if site["modality"] == "image":
            IMAGE_UPDATE = True
            continue
        else: pass

    if not FUSION_UPDATE:
        if site["modality"] == "multi":
            FUSION_UPDATE = True
            continue
        else: pass

In [ ]:
# 4.3 site-level evaluations
all_site_eva_results = []

for site in sites:
    site_level_eva_results = {}
    # 1. make predictions -> get df for predictions: y_true, y_pred, y_prob,  
    predict_df = site_prediction(global_state, site)

    # 2. calculation 5 metrics
    site_cm    = site_cm_calculation   (predict_df)
    site_auroc = site_auroc_calculation(predict_df)
    site_auprc = site_auprc_calculation(predict_df)
    site_acc   = site_acc_calculation  (predict_df)
    site_f1    = site_f1_calculation   (predict_df)

    # 3. record the results
    # basic info
    site_level_eva_results["name"]      = site["name"]
    site_level_eva_results["modality"]  = site["modality"]
    site_level_eva_results["n_classes"] = site["n_classes"]
    # 5 meritcs
    site_level_eva_results["site_cm"]    = site_cm  
    site_level_eva_results["site_auroc"] = site_auroc
    site_level_eva_results["site_auprc"] = site_auprc
    site_level_eva_results["site_acc"]   = site_acc
    site_level_eva_results["site_f1"]    = site_f1

    all_site_eva_results.append(site_level_eva_results)

global update round 1:
Start site traininig
tabular_1: tabular site training START
 [best updated] acc: 0.8342
 [best updated] acc: 0.8359
tabular_1: tabular site training DONE
image_1: image site training START
 [best updated] acc: 0.6392
 [best updated] acc: 0.6456
 [best updated] acc: 0.6603
 [best updated] acc: 0.6835
image_1: image site training DONE
Finish site traininig
Start head re-training
tabular_1: tabular site training START
 [best updated] acc: 0.8358
 [best updated] acc: 0.8360
tabular_1: tabular site training DONE
image_1: image site training START
 [best updated] acc: 0.6793
 [best updated] acc: 0.6772
image_1: image site training DONE
Finish head re-training

global update round 2:
Start site traininig
tabular_1: tabular site training START
 [best updated] acc: 0.8355
 [best updated] acc: 0.8356
 [best updated] acc: 0.8357
tabular_1: tabular site training DONE
image_1: image site training START
 [best updated] acc: 0.7004
 [best updated] acc: 0.7046
image_1: image sit

In [ ]:
# 4.4 "site-group" level evaluation
# 4.4.1 Build Grouping DataFrame
df_sites = pd.DataFrame(all_site_eva_results)
metric_cols = ["site_auroc", "site_auprc", "site_acc", "site_f1"]
info_cols   = ["name", "modality", "n_classes"]

df_sites = df_sites[info_cols + metric_cols]

# Create a "group key" for each site-group (modality + label space)
df_sites["group_key"] = df_sites["modality"].astype(str) + "_" + df_sites["n_classes"].astype(str)

print(f"[Grouping] Total sites: {len(df_sites)}")
print(f"[Grouping] Found {df_sites['group_key'].nunique()} unique site-groups.\n")

In [ ]:
# 4.4.2 Group-level Evaluation
grouped = df_sites.groupby(["modality", "n_classes"])
group_level_results = []

for (mod, ncls), subdf in grouped:
    print(f"\n[Group Evaluation] Modality={mod}, Classes={ncls} (Sites={len(subdf)})")

    group_metrics = {}
    group_metrics["modality"]  = mod
    group_metrics["n_classes"] = ncls
    group_metrics["n_sites"]   = len(subdf)

    # For each metric: macro(mean), micro(weighted mean, currently same as macro since no sample size info), std, range
    for metric in metric_cols:
        values = subdf[metric].values.astype(float)
        group_metrics[f"{metric}_macro"] = np.mean(values)
        group_metrics[f"{metric}_micro"] = np.mean(values)  # placeholder; can weight by sample size if available
        group_metrics[f"{metric}_std"]   = np.std(values)
        group_metrics[f"{metric}_range"] = np.max(values) - np.min(values)

    group_level_results.append(group_metrics)

# Build summary dataframe
df_group_summary = pd.DataFrame(group_level_results)

print("\n===== Site-group Summary =====")
print(df_group_summary[["modality", "n_classes", "n_sites",
                        "site_auroc_macro", "site_auprc_macro", "site_acc_macro", "site_f1_macro"]])
